<a href="https://colab.research.google.com/github/dakshini01/Statistical-Learning-e20181/blob/main/Assignment_7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

This notebook answers all tasks in the assignment section titled **Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates**.

The structural remaining-stiffness efficiency factor is

$$
\Theta=\theta,\qquad \theta\in(0,1].
$$

Here, $\theta=1$ represents a perfectly healthy structure and $\theta\to0$ represents severe degradation. The sensor model is

$$
Y_k=\theta K_{\mathrm{nominal}}e^{\epsilon_k},
\qquad
\epsilon_k\sim\mathcal{N}(0,\sigma^2).
$$


## 1. Prior Belief Boundaries

The initial prior is

$$
\Theta\sim\operatorname{Beta}(8,1.5).
$$

Its density is

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{B(8,1.5)}\theta^{7}(1-\theta)^{0.5},
\qquad 0<\theta<1.
$$

The expected prior stiffness efficiency is

$$
\mathbb{E}[\Theta^{(0)}]
=
\frac{8}{8+1.5}
=
\frac{8}{9.5}
\approx0.8421.
$$

Thus,

$$
\boxed{\mathbb{E}[\Theta^{(0)}]\approx0.8421}
$$

This is an appropriate prior for a component assumed to be healthy because it places most probability mass near the upper end of the physical interval, while still allowing uncertainty and possible degradation.


In [1]:
import numpy as np
from scipy.stats import beta
import plotly.graph_objects as go

theta_grid = np.linspace(0.01, 1.0, 1000)
alpha_0, beta_0 = 8.0, 1.5

prior = beta.pdf(theta_grid, alpha_0, beta_0)
prior /= np.trapezoid(prior, theta_grid)
prior_mean = alpha_0 / (alpha_0 + beta_0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior, mode="lines", name="Beta(8, 1.5) prior"))
fig.add_vline(x=prior_mean, line_dash="dash", annotation_text=f"Prior mean = {prior_mean:.4f}")
fig.update_layout(
    title="Initial Prior for Structural Stiffness Efficiency",
    xaxis_title="Remaining stiffness efficiency θ",
    yaxis_title="Prior density",
    template="plotly_white"
)
fig.show()

print(f"Expected prior stiffness efficiency = {prior_mean:.6f}")


Expected prior stiffness efficiency = 0.842105


## 2. Structural Likelihood Formulation

The measurement model is

$$
Y_k=\theta K_{\mathrm{nominal}}e^{\epsilon_k},
\qquad \epsilon_k\sim\mathcal{N}(0,\sigma^2).
$$

Taking logarithms,

$$
\log Y_k=\log(\theta K_{\mathrm{nominal}})+\epsilon_k.
$$

Therefore,

$$
\log Y_k\mid\Theta=\theta
\sim
\mathcal{N}\left(\log(\theta K_{\mathrm{nominal}}),\sigma^2\right).
$$

Hence the likelihood of one positive sensor reading $y_k$ is

$$
\boxed{
L(y_k\mid\theta)
=
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{\left(\log y_k-\log(\theta K_{\mathrm{nominal}})\right)^2}{2\sigma^2}
\right]
}
$$

for $y_k>0$ and $\theta\in(0,1]$.

For the running history $y^{(k)}=(y_1,\ldots,y_k)$, conditional independence gives

$$
\boxed{
L(y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\frac{1}{y_i\sigma\sqrt{2\pi}}
\exp\left[
-\frac{\left(\log y_i-\log(\theta K_{\mathrm{nominal}})\right)^2}{2\sigma^2}
\right]
}
$$


## 3. Mathematical Formulation of the Non-Conjugate Grid Update

The Beta prior has kernel

$$
\theta^{\alpha_0-1}(1-\theta)^{\beta_0-1},
$$

but the likelihood depends on $\log\theta$ through

$$
\exp\left[-\frac{(\log y_k-\log(\theta K_{\mathrm{nominal}}))^2}{2\sigma^2}\right].
$$

Their product is not the kernel of another Beta distribution. Therefore, the model is non-conjugate and the posterior has no simple closed-form Beta update.

The recursive posterior update is

$$
\boxed{
f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})
\propto
L(y_k\mid\theta)
f_{\Theta\mid Y^{(k-1)}}(\theta\mid y^{(k-1)})
}
$$

and the normalized form is

$$
\boxed{
f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})
=
\frac{L(y_k\mid\theta)f_{\Theta\mid Y^{(k-1)}}(\theta\mid y^{(k-1)})}
{\int_0^1 L(y_k\mid s)f_{\Theta\mid Y^{(k-1)}}(s\mid y^{(k-1)})\,ds}
}
$$

The computational grid uses $\theta\in[0.01,1.0]$ so that $\log\theta$ is always defined.


## 4. Running Point Estimates

The running posterior mean is

$$
\boxed{
\widehat{\theta}^{(k)}_{\mathrm{Bayes}}
=
\int_{0.01}^{1.0}
\theta f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})\,d\theta
}
$$

and the running MAP estimate is

$$
\boxed{
\widehat{\theta}^{(k)}_{\mathrm{MAP}}
=
\underset{\theta\in[0.01,1.0]}{\arg\max}\;
f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})
}
$$

On a discrete grid,

$$
\widehat{\theta}^{(k)}_{\mathrm{Bayes}}
\approx
\operatorname{trapezoid}(\theta P_k,\theta),
$$

and

$$
\widehat{\theta}^{(k)}_{\mathrm{MAP}}
=
\theta_{\arg\max_m P_k(\theta_m)}.
$$


## 5. Algorithmic Grid Approximation and Normalization

Choose $M$ equally spaced values over the bounded interval:

$$
\theta_m=0.01+(m-1)\Delta\theta,
\qquad
\Delta\theta=\frac{1.0-0.01}{M-1}.
$$

The lower bound is $0.01$, not $0$, because the likelihood contains $\log\theta$.

At step $k$:

1. Evaluate the likelihood $L(y_k\mid\theta_m)$.
2. Form the unnormalized posterior

$$
\widetilde{P}_k(\theta_m)=P_{k-1}(\theta_m)L(y_k\mid\theta_m).
$$

3. Compute the trapezoidal normalization constant

$$
Z_k\approx\operatorname{trapezoid}(\widetilde{P}_k,\theta).
$$

Equivalently,

$$
Z_k\approx\sum_{m=1}^{M-1}
\frac{\widetilde{P}_k(\theta_m)+\widetilde{P}_k(\theta_{m+1})}{2}\Delta\theta.
$$

4. Normalize

$$
P_k(\theta_m)=\frac{\widetilde{P}_k(\theta_m)}{Z_k}.
$$

5. Compute the posterior mean and MAP from the normalized grid.


## 6. Performance Tracking and Degradation Convergence Analysis

Use

$$
\theta_{\mathrm{true}}=0.68,
\qquad
n=15,
\qquad
K_{\mathrm{nominal}}=50.0,
\qquad
\sigma=0.15.
$$

The code below simulates the sensor stream, performs sequential bounded-grid updates, plots posterior densities at $k\in\{0,1,2,5,10,15\}$, and tracks the posterior mean and MAP estimate.


In [2]:
import numpy as np
from scipy.stats import beta
import plotly.graph_objects as go

rng = np.random.default_rng(24)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_readings = 15

theta_grid = np.linspace(0.01, 1.0, 2000)
milestones = {0, 1, 2, 5, 10, 15}


def lognormal_likelihood(y, theta, K_nominal, sigma):
    if y <= 0:
        raise ValueError("Sensor readings must be positive.")

    scale = theta * K_nominal
    return (
        1.0 / (y * sigma * np.sqrt(2.0 * np.pi))
        * np.exp(-((np.log(y) - np.log(scale)) ** 2) / (2.0 * sigma**2))
    )


def grid_quantile(grid, density, probability):
    increments = 0.5 * (density[:-1] + density[1:]) * np.diff(grid)
    cdf = np.concatenate(([0.0], np.cumsum(increments)))
    cdf /= cdf[-1]
    return np.interp(probability, cdf, grid)


# Initial prior
posterior = beta.pdf(theta_grid, a=8.0, b=1.5)
posterior /= np.trapezoid(posterior, theta_grid)

steps = list(range(n_readings + 1))
posterior_means = [np.trapezoid(theta_grid * posterior, theta_grid)]
map_estimates = [theta_grid[np.argmax(posterior)]]
lower_95 = [grid_quantile(theta_grid, posterior, 0.025)]
upper_95 = [grid_quantile(theta_grid, posterior, 0.975)]
posterior_snapshots = {0: posterior.copy()}
sensor_readings = []

for k in range(1, n_readings + 1):
    epsilon_k = rng.normal(0.0, sigma)
    y_k = theta_true * K_nominal * np.exp(epsilon_k)
    sensor_readings.append(y_k)

    likelihood = lognormal_likelihood(
        y=y_k,
        theta=theta_grid,
        K_nominal=K_nominal,
        sigma=sigma
    )

    unnormalized = posterior * likelihood
    normalizer = np.trapezoid(unnormalized, theta_grid)

    if normalizer <= 0 or not np.isfinite(normalizer):
        raise FloatingPointError(f"Normalization failed at step {k}.")

    posterior = unnormalized / normalizer

    posterior_means.append(np.trapezoid(theta_grid * posterior, theta_grid))
    map_estimates.append(theta_grid[np.argmax(posterior)])
    lower_95.append(grid_quantile(theta_grid, posterior, 0.025))
    upper_95.append(grid_quantile(theta_grid, posterior, 0.975))

    if k in milestones:
        posterior_snapshots[k] = posterior.copy()

# Posterior density progression
fig1 = go.Figure()
for k in sorted(posterior_snapshots):
    fig1.add_trace(go.Scatter(
        x=theta_grid,
        y=posterior_snapshots[k],
        mode="lines",
        name=f"Step {k}"
    ))

fig1.add_vline(x=theta_true, line_dash="dash", annotation_text=f"True state θ={theta_true}")
fig1.update_layout(
    title="Posterior Density Progression",
    xaxis_title="Remaining stiffness efficiency θ",
    yaxis_title="Posterior density",
    template="plotly_white",
    hovermode="x unified"
)
fig1.show()

# Estimator timeline
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=steps, y=posterior_means, mode="lines+markers", name="Posterior mean"))
fig2.add_trace(go.Scatter(x=steps, y=map_estimates, mode="lines+markers", name="MAP estimate"))
fig2.add_trace(go.Scatter(x=steps, y=upper_95, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig2.add_trace(go.Scatter(x=steps, y=lower_95, mode="lines", fill="tonexty", line=dict(width=0), name="95% credible interval"))
fig2.add_hline(y=theta_true, line_dash="dash", annotation_text=f"True state θ={theta_true}")
fig2.update_layout(
    title="Convergence of Structural Health Estimates",
    xaxis_title="Number of sensor readings k",
    yaxis_title="Estimated stiffness efficiency",
    template="plotly_white",
    hovermode="x unified"
)
fig2.show()

# Operational definition of "confidently isolate":
# the 95% credible interval contains theta_true and has width <= 0.15.
confidence_step = None
for k in range(1, n_readings + 1):
    contains_true = lower_95[k] <= theta_true <= upper_95[k]
    narrow_enough = (upper_95[k] - lower_95[k]) <= 0.15
    if contains_true and narrow_enough:
        confidence_step = k
        break

print("Sensor readings:")
for k, y_k in enumerate(sensor_readings, start=1):
    print(f"Step {k:2d}: y_k = {y_k:.4f}")

print()
print(f"Final posterior mean = {posterior_means[-1]:.4f}")
print(f"Final MAP estimate   = {map_estimates[-1]:.4f}")
print(f"Final 95% credible interval = [{lower_95[-1]:.4f}, {upper_95[-1]:.4f}]")
print(f"True state = {theta_true:.4f}")

if confidence_step is None:
    print("The chosen confidence criterion was not met within 15 readings.")
else:
    print(f"The chosen confidence criterion was first met after {confidence_step} readings.")


Sensor readings:
Step  1: y_k = 41.6363
Step  2: y_k = 35.7956
Step  3: y_k = 28.5573
Step  4: y_k = 33.0591
Step  5: y_k = 32.3121
Step  6: y_k = 32.8585
Step  7: y_k = 37.1844
Step  8: y_k = 28.0645
Step  9: y_k = 39.3053
Step 10: y_k = 28.7077
Step 11: y_k = 33.0523
Step 12: y_k = 38.8384
Step 13: y_k = 37.5590
Step 14: y_k = 30.6523
Step 15: y_k = 44.3313

Final posterior mean = 0.6970
Final MAP estimate   = 0.6954
Final 95% credible interval = [0.6457, 0.7513]
True state = 0.6800
The chosen confidence criterion was first met after 8 readings.


## Analysis

The initial $\operatorname{Beta}(8,1.5)$ prior is optimistic because its mean is approximately $0.8421$, whereas the true remaining stiffness is $0.68$. The incoming log-normal sensor readings therefore need to overcome an initially healthy prior.

As the number of readings increases, the posterior mean and MAP estimate move toward $\theta_{\mathrm{true}}=0.68$, while the posterior density becomes narrower. This narrowing represents reduced uncertainty and increasing confidence in the estimated damage state.

The assignment does not give a precise mathematical definition of “confidently isolate,” so the exact number of required readings is not uniquely determined. The code explicitly defines one operational criterion: the $95\%$ credible interval must contain $0.68$ and have width no greater than $0.15$. The resulting step depends on the random sensor stream and this chosen criterion.

For structural safety decisions, the full posterior is more informative than a point estimate. Given a critical threshold $\theta_{\mathrm{critical}}$, engineers can compute

$$
P(\Theta<\theta_{\mathrm{critical}}\mid y^{(k)})
=
\int_{0.01}^{\theta_{\mathrm{critical}}}
f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})\,d\theta.
$$

A narrow posterior concentrated below a safety threshold is strong evidence that intervention is required.


In [3]:
# Optional posterior probability below a safety threshold

theta_critical = 0.75
mask = theta_grid <= theta_critical
probability_below_threshold = np.trapezoid(
    posterior[mask],
    theta_grid[mask]
)

print(
    f"P(Theta < {theta_critical:.2f} | data) = "
    f"{probability_below_threshold:.4f}"
)


P(Theta < 0.75 | data) = 0.9721
